# Silver Processing — Date

Cleans `ecommerce.bronze.bronze_date` (95 rows) into `ecommerce.silver.silver_date`:

* **Recomputed from `date`** (review found the raw derived attributes unreliable):
  * `day_name` — raw had mixed casing (`friday`, `SATURDAY`) and 2 wrong values; recomputed as Title Case (`Friday`)
  * `quarter` — 2 wrong values (e.g. quarter 2 for an August date); recomputed
  * `week_of_year` — all 95 raw values negative, 2 wrong even ignoring sign; recomputed as a positive ISO week
  * `year` — recomputed for consistency (raw values were already correct)
* **Duplicate rows removed** — 3 dates appeared twice (`2025-08-29`, `2025-09-25`, `2025-10-13`); safe to dedupe because attributes are recomputed
* `date` itself verified correctly parsed from the raw `dd-mm-yyyy` source
* **Metadata columns** (`source_file`, `ingested_at`) pass through untouched

In [0]:
from pyspark.sql.functions import col, year, quarter, weekofyear, date_format

bronze_table = "ecommerce.bronze.bronze_date"
target_table = "ecommerce.silver.silver_date"

df_bronze_date = spark.table(bronze_table)

# The date itself parsed correctly; its derived attributes did not.
# Recompute them from the date so every row is self-consistent,
# which also makes duplicate rows identical before dedup.
df_silver_date = (
    df_bronze_date
    .withColumn("year", year(col("date")))
    .withColumn("day_name", date_format(col("date"), "EEEE"))  # Title Case day name
    .withColumn("quarter", quarter(col("date")))
    .withColumn("week_of_year", weekofyear(col("date")))  # positive ISO week
    .dropDuplicates(["date"])  # one row per calendar day
)

# Idempotent full load into the silver table
(
    df_silver_date.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

print(f"Wrote {spark.table(target_table).count()} rows to {target_table}")

In [0]:
display(spark.table("ecommerce.silver.silver_date").limit(10))

In [0]:
# Check 1: one row per date, all attributes consistent with the date itself
spark.sql("""
    SELECT
      COUNT(*) AS rows,
      COUNT(DISTINCT date) AS distinct_dates,
      COUNT_IF(year != year(date)) AS year_mismatch,
      COUNT_IF(quarter != quarter(date)) AS quarter_mismatch,
      COUNT_IF(day_name != date_format(date, 'EEEE')) AS dayname_mismatch,
      COUNT_IF(week_of_year != weekofyear(date)) AS week_mismatch,
      COUNT_IF(week_of_year < 1) AS non_positive_weeks
    FROM ecommerce.silver.silver_date
""").show()

# Check 2: value ranges — Aug-Oct 2025 should be quarters 3-4 and weeks 31-45
spark.sql("""
    SELECT
      MIN(date) AS min_date, MAX(date) AS max_date,
      MIN(quarter) AS min_q, MAX(quarter) AS max_q,
      MIN(week_of_year) AS min_week, MAX(week_of_year) AS max_week,
      COUNT(DISTINCT day_name) AS distinct_days
    FROM ecommerce.silver.silver_date
""").show()